# 0.2 bis · Extraction des attributs OSM

Ce notebook constitue un GeoPackage d'attributs OSM dans une aire d'étude.

Principes retenus :
- le territoire sert uniquement à résoudre l'AOI et les chemins de fichiers ;
- il n'y a pas de paramètre `network`, car les couches exportées sont des attributs OSM ;
- les couches `piste`, `bande` et `vitesse` réutilisent `bike_edges_graph.geojson` si disponible, afin d'éviter des requêtes OSM inutiles ;
- si cette sortie préparée n'est pas disponible, les mêmes couches peuvent être extraites directement depuis l'API OSM.

In [ ]:
from __future__ import annotations

import json
import re
import warnings
from pathlib import Path

import geopandas as gpd
import osmnx as ox
import pandas as pd
from shapely.ops import unary_union

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

## 1. Paramètres structurants

Les paramètres ci-dessous définissent le périmètre de travail et la stratégie de chargement. Les chemins et fichiers dérivés sont résolus dans la cellule suivante.

In [ ]:
TERRITORY = "GG"  # "GG" ou "GE"
AOI_MODE = "shapefile"  # "shapefile" ou "geocode"

OPERATION_CRS = "EPSG:2056"  # CRS métrique pour calculs éventuels
EXPORT_CRS = "EPSG:4326"  # CRS final des couches exportées

# Autoriser les requêtes OSM nécessaires aux couches extraites directement.
ALLOW_OSM_API_REQUESTS = True

# Réutiliser les sorties du notebook 0_1 pour les couches piste/bande/vitesse.
# Si le fichier manque ou si une couche est vide, ce fallback OSM peut prendre le relais.
USE_PREPARED_BIKE_EDGES = True
ALLOW_OSM_API_FALLBACK = False

# Export GeoPackage : True repart d'un fichier propre à chaque exécution complète.
OVERWRITE_GPKG = True

# Les conflits marche/vélo ne sont pas recalculés ici ; ils sont exportés seulement si la couche existe déjà.
INCLUDE_PREPARED_CONFLICT_ZONES = True

## 2. Configuration du territoire

`TERRITORY` est une clé de configuration. Le périmètre spatial effectif vient de `AOI_MODE` : shapefile si `AOI_MODE = "shapefile"`, géocodage des lieux si `AOI_MODE = "geocode"`.

In [ ]:
INPUT_ROOT = Path("../../Data/input")

TERRITORY_CONFIG = {
    "GG": {
        "network_dir": INPUT_ROOT / "networkGG",
        "aoi_shapefile": INPUT_ROOT / "network_agreg/AGGLO_PERIMETRE_AVEC_LAC-SHP/AGGLO_PERIMETRE_AVEC_LAC.shp",
        "osm_attr_gpkg": INPUT_ROOT / "attributs/GG/osm/osm_attributes.gpkg",
        "places": [
            "Genève, Switzerland",
            "Vandoeuvres, Switzerland",
            "Cologny, Switzerland",
            "Collonge-Bellerive, Switzerland",
            "Choulex, Switzerland",
        ],
    },
    "GE": {
        "network_dir": INPUT_ROOT / "networkGE",
        "aoi_shapefile": INPUT_ROOT / "network_agreg/CAD_LIMITE_CANTON-SHP/CAD_LIMITE_CANTON_POLYGON.shp",
        "osm_attr_gpkg": INPUT_ROOT / "attributs/GE/osm/osm_attributes.gpkg",
        "places": [
            "Genève, Switzerland",
            "Vandoeuvres, Switzerland",
            "Cologny, Switzerland",
            "Collonge-Bellerive, Switzerland",
            "Choulex, Switzerland",
        ],
    },
}

if TERRITORY not in TERRITORY_CONFIG:
    raise ValueError(f"Territoire inconnu: {TERRITORY}. Valeurs attendues: {list(TERRITORY_CONFIG)}")

cfg = TERRITORY_CONFIG[TERRITORY]
network_dir = cfg["network_dir"]
aoi_shapefile = cfg["aoi_shapefile"]
places = cfg["places"]
gpkg_attr = cfg["osm_attr_gpkg"]

gpkg_attr.parent.mkdir(parents=True, exist_ok=True)
if OVERWRITE_GPKG and gpkg_attr.exists():
    gpkg_attr.unlink()

print(f"Territoire: {TERRITORY}")
print(f"AOI mode: {AOI_MODE}")
print(f"Dossier réseau préparé: {network_dir}")
print(f"GeoPackage de sortie: {gpkg_attr}")

## 3. AOI et couches préparées optionnelles

Les couches préparées sont chargées seulement si elles existent. Elles servent d'abord à limiter les requêtes OSM pour `piste`, `bande` et `vitesse`.

In [ ]:
def load_aoi(aoi_mode: str, shapefile_path: Path, geocode_places: list[str]) -> gpd.GeoDataFrame:
    """Retourne une GeoDataFrame EPSG:4326 contenant un polygone d'AOI."""
    if aoi_mode == "shapefile":
        gdf = gpd.read_file(shapefile_path)
        gdf = gdf.to_crs(EXPORT_CRS) if gdf.crs is not None else gdf.set_crs(EXPORT_CRS)
        geom = unary_union(gdf.geometry.values)
        return gpd.GeoDataFrame({"name": ["aoi"]}, geometry=[geom], crs=EXPORT_CRS)

    if aoi_mode == "geocode":
        polygons = []
        for place in geocode_places:
            place_gdf = ox.geocode_to_gdf(place).to_crs(EXPORT_CRS)
            polygons.append(unary_union(place_gdf.geometry.values))
        geom = unary_union(polygons)
        return gpd.GeoDataFrame({"name": ["aoi"]}, geometry=[geom], crs=EXPORT_CRS)

    raise ValueError(f"AOI_MODE inconnu: {aoi_mode}")


def read_optional_gdf(path: Path, label: str) -> gpd.GeoDataFrame | None:
    """Charge une couche si elle existe, sinon retourne None sans bloquer le notebook."""
    if not path.exists():
        print(f"{label}: fichier absent ({path})")
        return None
    try:
        gdf = gpd.read_file(path)
    except Exception as exc:
        print(f"{label}: lecture impossible ({exc})")
        return None
    print(f"{label}: {len(gdf):,} objets chargés")
    return gdf


aoi = load_aoi(AOI_MODE, aoi_shapefile, places)
aoi_poly = aoi.geometry.iloc[0]

bike_edges_gdf_graph = None
if USE_PREPARED_BIKE_EDGES:
    bike_edges_gdf_graph = read_optional_gdf(network_dir / "bike_edges_graph.geojson", "bike_edges_graph")

conflict_zones = None
if INCLUDE_PREPARED_CONFLICT_ZONES:
    conflict_zones = read_optional_gdf(network_dir / "pedestrian_bike_conflicts.geojson", "pedestrian_bike_conflicts")

aoi

## 4. Fonctions utilitaires

In [ ]:
MIN_COLS = ["geometry", "osmid"]
LAYER_LOG: list[dict] = []


def empty_feature_gdf(keep_cols: list[str] | None = None) -> gpd.GeoDataFrame:
    keep_cols = keep_cols or []
    cols = list(dict.fromkeys(MIN_COLS + keep_cols))
    return gpd.GeoDataFrame(columns=cols, geometry="geometry", crs=EXPORT_CRS)


def false_mask(gdf: gpd.GeoDataFrame) -> pd.Series:
    return pd.Series(False, index=gdf.index)


def col_notna(gdf: gpd.GeoDataFrame, col: str) -> pd.Series:
    if col not in gdf.columns:
        return false_mask(gdf)
    return gdf[col].notna()


def col_eq(gdf: gpd.GeoDataFrame, col: str, value: str) -> pd.Series:
    if col not in gdf.columns:
        return false_mask(gdf)
    return gdf[col].astype("string").str.lower().eq(value.lower()).fillna(False)


def col_isin(gdf: gpd.GeoDataFrame, col: str, values: list[str] | set[str]) -> pd.Series:
    if col not in gdf.columns:
        return false_mask(gdf)
    expected = {str(v).lower() for v in values}
    return gdf[col].astype("string").str.lower().isin(expected).fillna(False)


def col_contains_any(gdf: gpd.GeoDataFrame, col: str, tokens: list[str] | set[str]) -> pd.Series:
    if col not in gdf.columns:
        return false_mask(gdf)
    pattern = "|".join(re.escape(str(token).lower()) for token in tokens)
    return gdf[col].astype("string").str.lower().str.contains(pattern, na=False)


def any_col_contains(gdf: gpd.GeoDataFrame, cols: list[str], tokens: list[str] | set[str]) -> pd.Series:
    mask = false_mask(gdf)
    for col in cols:
        mask = mask | col_contains_any(gdf, col, tokens)
    return mask


def true_tag():
    """Pour OSMnx, True signifie : récupérer les objets où le tag existe."""
    return True


def ensure_osmid(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    if "osmid" in gdf.columns:
        return gdf
    gdf = gdf.copy()
    for candidate in ["id", "osm_id", "@id", "index"]:
        if candidate in gdf.columns:
            gdf["osmid"] = gdf[candidate]
            return gdf
    gdf["osmid"] = range(len(gdf))
    return gdf


def extract_osm_features(aoi_polygon, tags: dict, geom_types=("Point",), keep_cols=None) -> gpd.GeoDataFrame:
    keep_cols = keep_cols or []
    if not ALLOW_OSM_API_REQUESTS:
        return empty_feature_gdf(keep_cols)

    try:
        gdf = ox.features_from_polygon(aoi_polygon, tags).reset_index()
    except Exception as exc:
        print(f"OSM extract error for {tags}: {exc}")
        return empty_feature_gdf(keep_cols)

    if gdf.empty:
        return empty_feature_gdf(keep_cols)

    gdf = gdf[gdf.geometry.notna()].copy()
    gdf = gdf[gdf.geometry.type.isin(list(geom_types))].copy()
    if gdf.empty:
        return empty_feature_gdf(keep_cols)

    gdf = ensure_osmid(gdf)
    cols = ["geometry", "osmid"] + [col for col in keep_cols if col in gdf.columns]
    cols = list(dict.fromkeys(cols))
    return gdf[cols].set_crs(EXPORT_CRS, allow_override=True)


def keep_columns(gdf: gpd.GeoDataFrame, keep_cols: list[str]) -> gpd.GeoDataFrame:
    out_cols = ["geometry"]
    for id_col in ["osmid", "id", "fid", "edge_id"]:
        if id_col in gdf.columns:
            out_cols.append(id_col)
            break
    out_cols += [col for col in keep_cols if col in gdf.columns]
    out_cols = list(dict.fromkeys(out_cols))
    return gdf[out_cols].copy()


def normalize_object_columns(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """Sérialise les valeurs non scalaires pour éviter les erreurs d'export GPKG."""
    out = gdf.copy()
    for col in out.columns:
        if col == out.geometry.name or out[col].dtype != "object":
            continue
        out[col] = out[col].map(
            lambda value: json.dumps(list(value), ensure_ascii=False)
            if isinstance(value, (list, tuple, set))
            else json.dumps(value, ensure_ascii=False)
            if isinstance(value, dict)
            else value
        )
    return out


def export_layer_gpkg(gdf: gpd.GeoDataFrame | None, layer: str, source: str) -> None:
    if gdf is None or gdf.empty:
        LAYER_LOG.append({"layer": layer, "source": source, "n": 0, "exported": False})
        print(f"{layer}: 0 objet ({source})")
        return

    export_gdf = normalize_object_columns(gdf).to_crs(EXPORT_CRS)
    mode = "a" if gpkg_attr.exists() else "w"
    export_gdf.to_file(gpkg_attr, layer=layer, driver="GPKG", mode=mode)
    LAYER_LOG.append({"layer": layer, "source": source, "n": len(export_gdf), "exported": True})
    print(f"{layer}: {len(export_gdf):,} objets exportés ({source})")

## 5. Définitions des couches OSM

Ces dictionnaires définissent les tags à extraire et les colonnes conservées. Ils restent indépendants d'un usage final marche ou vélo.

In [ ]:
POI_VELO_DEFINITIONS = {
    "borne_reparation": {
        "tags": {"amenity": "bicycle_repair_station"},
        "keep_cols": ["amenity", "access"],
    },
    "stationnement_velo": {
        "tags": {"amenity": "bicycle_parking"},
        "keep_cols": ["amenity", "access", "capacity", "covered", "bicycle_parking"],
    },
    "location": {
        "tags": {"amenity": "bicycle_rental"},
        "keep_cols": ["amenity", "operator"],
    },
    "service_velo": {
        "tags": {"shop": "bicycle"},
        "keep_cols": [
            "shop",
            "name",
            "operator",
            "service:bicycle:repair",
            "service:bicycle:sales",
            "service:bicycle:rental",
            "service:bicycle:pump",
        ],
    },
}

POI_PEDESTRIAN_DEFINITIONS = {
    "fontaine": {
        "tags": {"amenity": ["drinking_water", "fountain"]},
        "keep_cols": ["amenity", "name", "operator", "access"],
    },
    "banc": {
        "tags": {"amenity": "bench"},
        "keep_cols": ["amenity", "backrest", "covered"],
    },
    "toilette": {
        "tags": {"amenity": "toilets"},
        "keep_cols": ["amenity", "access", "wheelchair", "fee", "opening_hours"],
    },
}

POI_TP_DEFINITION = {
    "transport_public": {
        "tags": {
            "highway": ["bus_stop"],
            "railway": ["station", "halt", "tram_stop"],
            "amenity": ["bus_station"],
            "public_transport": ["station"],
        },
        "keep_cols": [
            "highway",
            "railway",
            "amenity",
            "public_transport",
            "name",
            "operator",
            "network",
            "ref",
            "wheelchair",
            "shelter",
            "bench",
        ],
    }
}

POI_AMENITES_DEFINITIONS = {
    "amenite": {
        "tags": {"amenity": true_tag()},
        "keep_cols": ["amenity", "name", "operator", "access"],
    }
}

OSM_NODE_FEATURES = {
    "crossing": {
        "tags": {"crossing": true_tag()},
        "keep_cols": ["crossing", "crossing:markings", "crossing:signals"],
    },
    "traffic_signals": {
        "tags": {"highway": "traffic_signals"},
        "keep_cols": ["highway", "button_operated", "tactile_paving"],
    },
    "barrier": {
        "tags": {"barrier": true_tag()},
        "keep_cols": ["barrier", "access", "bicycle", "wheelchair"],
    },
    "traffic_calming_point": {
        "tags": {"traffic_calming": true_tag()},
        "keep_cols": ["traffic_calming"],
    },
}

OSM_WAY_FEATURES = {
    "chemin": {"tags": {"highway": ["path"]}, "keep_cols": ["highway"]},
    "revetement": {"tags": {"surface": true_tag()}, "keep_cols": ["surface"]},
    "etat_chaussee": {"tags": {"smoothness": true_tag()}, "keep_cols": ["smoothness"]},
    "eclairage": {"tags": {"lit": true_tag()}, "keep_cols": ["lit"]},
    "largeur": {"tags": {"width": true_tag()}, "keep_cols": ["width"]},
    "pente": {"tags": {"incline": true_tag()}, "keep_cols": ["incline"]},
    "giratoire": {"tags": {"junction": "roundabout"}, "keep_cols": ["junction"]},
}

OSM_POLY_FEATURES = {
    "canopee": {
        "tags": {"landuse": ["forest"], "natural": ["wood"]},
        "keep_cols": ["landuse", "natural", "name"],
    },
    "lac_cours_deau": {
        "tags": {
            "natural": ["water"],
            "water": ["river", "lake", "pond", "reservoir", "lagoon"],
            "waterway": ["riverbank"],
        },
        "keep_cols": ["natural", "water", "name", "waterway"],
    },
    "espaces_ouverts": {
        "tags": {
            "leisure": ["park", "playground"],
            "landuse": ["recreation_ground"],
            "amenity": ["grave_yard"],
        },
        "keep_cols": ["leisure", "landuse", "amenity", "name", "access", "operator", "ownership"],
    },
    "landuse_env": {
        "tags": {
            "landuse": [
                "forest",
                "grass",
                "meadow",
                "recreation_ground",
                "residential",
                "cemetery",
                "allotments",
                "commercial",
                "retail",
                "industrial",
                "construction",
                "railway",
            ]
        },
        "keep_cols": ["landuse", "name"],
    },
}

## 6. Couches cyclables linéaires : source préparée puis fallback OSM

`bike_edges_graph.geojson` est la source préférée. Les filtres s'appuient d'abord sur `infra_bike`, puis sur les tags OSM conservés dans la couche préparée.

In [ ]:
CYCLEWAY_TAG_COLS = ["cycleway", "cycleway:left", "cycleway:right", "cycleway:both"]


def filter_piste(gdf: gpd.GeoDataFrame) -> pd.Series:
    return (
        col_isin(gdf, "infra_bike", {"piste_cyclable"})
        | col_eq(gdf, "highway", "cycleway")
        | any_col_contains(gdf, CYCLEWAY_TAG_COLS, {"track", "separated"})
    )


def filter_bande(gdf: gpd.GeoDataFrame) -> pd.Series:
    return (
        col_isin(gdf, "infra_bike", {"bande_cyclable"})
        | any_col_contains(gdf, CYCLEWAY_TAG_COLS, {"lane", "advisory", "shared"})
    )


def filter_vitesse(gdf: gpd.GeoDataFrame) -> pd.Series:
    return col_notna(gdf, "maxspeed")


CYCLING_LINEAR_LAYERS = {
    "piste": {
        "filter": filter_piste,
        "osm_tags": {
            "highway": "cycleway",
            "cycleway": true_tag(),
            "cycleway:left": true_tag(),
            "cycleway:right": true_tag(),
            "cycleway:both": true_tag(),
        },
        "keep_cols": [
            "infra_bike",
            "bike_direction",
            "source",
            "highway",
            "cycleway",
            "cycleway:left",
            "cycleway:right",
            "cycleway:both",
            "bicycle",
            "segregated",
            "oneway",
            "oneway:bicycle",
            "name",
            "maxspeed",
        ],
    },
    "bande": {
        "filter": filter_bande,
        "osm_tags": {
            "cycleway": true_tag(),
            "cycleway:left": true_tag(),
            "cycleway:right": true_tag(),
            "cycleway:both": true_tag(),
        },
        "keep_cols": [
            "infra_bike",
            "bike_direction",
            "source",
            "highway",
            "cycleway",
            "cycleway:left",
            "cycleway:right",
            "cycleway:both",
            "bicycle",
            "oneway",
            "oneway:bicycle",
            "name",
            "maxspeed",
        ],
    },
    "vitesse": {
        "filter": filter_vitesse,
        "osm_tags": {"maxspeed": true_tag()},
        "keep_cols": ["highway", "maxspeed", "source:maxspeed", "name"],
    },
}


def extract_from_prepared_bike_edges(layer: str, cfg: dict) -> gpd.GeoDataFrame | None:
    if bike_edges_gdf_graph is None or bike_edges_gdf_graph.empty:
        return None
    mask = cfg["filter"](bike_edges_gdf_graph)
    gdf = bike_edges_gdf_graph[mask].copy()
    if gdf.empty:
        return gdf
    return keep_columns(gdf, cfg["keep_cols"])


def extract_linear_fallback_from_osm(layer: str, cfg: dict) -> gpd.GeoDataFrame:
    gdf = extract_osm_features(
        aoi_poly,
        cfg["osm_tags"],
        geom_types=("LineString", "MultiLineString"),
        keep_cols=cfg["keep_cols"],
    )
    if gdf.empty:
        return gdf
    return keep_columns(gdf[cfg["filter"](gdf)].copy(), cfg["keep_cols"])

## 7. Export des points

In [ ]:
point_layers = {}
point_layers.update(POI_VELO_DEFINITIONS)
point_layers.update(POI_PEDESTRIAN_DEFINITIONS)
point_layers.update(POI_AMENITES_DEFINITIONS)
point_layers.update(OSM_NODE_FEATURES)

for layer, layer_cfg in point_layers.items():
    gdf = extract_osm_features(
        aoi_poly,
        layer_cfg["tags"],
        geom_types=("Point",),
        keep_cols=layer_cfg["keep_cols"],
    )
    export_layer_gpkg(keep_columns(gdf, layer_cfg["keep_cols"]), layer, "osm_api")

for layer, layer_cfg in POI_TP_DEFINITION.items():
    gdf = extract_osm_features(
        aoi_poly,
        layer_cfg["tags"],
        geom_types=("Point", "Polygon", "MultiPolygon"),
        keep_cols=layer_cfg["keep_cols"],
    )
    if not gdf.empty:
        poly_mask = gdf.geometry.type.isin(["Polygon", "MultiPolygon"])
        if poly_mask.any():
            gdf.loc[poly_mask, "geometry"] = gdf.loc[poly_mask].geometry.representative_point()
    export_layer_gpkg(keep_columns(gdf, layer_cfg["keep_cols"]), layer, "osm_api")


## 8. Export des couches cyclables linéaires

In [ ]:
for layer, layer_cfg in CYCLING_LINEAR_LAYERS.items():
    source = "prepared_bike_edges"
    gdf = extract_from_prepared_bike_edges(layer, layer_cfg) if USE_PREPARED_BIKE_EDGES else None

    if (gdf is None or gdf.empty) and ALLOW_OSM_API_REQUESTS and ALLOW_OSM_API_FALLBACK:
        source = "osm_api_fallback"
        gdf = extract_linear_fallback_from_osm(layer, layer_cfg)

    export_layer_gpkg(gdf, layer, source)

## 9. Export des lignes OSM génériques

In [ ]:
for layer, layer_cfg in OSM_WAY_FEATURES.items():
    gdf = extract_osm_features(
        aoi_poly,
        layer_cfg["tags"],
        geom_types=("LineString", "MultiLineString"),
        keep_cols=layer_cfg["keep_cols"],
    )
    export_layer_gpkg(keep_columns(gdf, layer_cfg["keep_cols"]), layer, "osm_api")

## 10. Export de l'eau et des polygones

`lac_cours_deau` combine les lignes hydrographiques et les surfaces d'eau dans une même couche.

In [ ]:
water_line_cfg = {
    "tags": {"waterway": ["river", "stream", "canal", "ditch", "flowline"]},
    "keep_cols": ["natural", "water", "name", "waterway"],
}
water_poly_cfg = OSM_POLY_FEATURES["lac_cours_deau"]

water_lines = extract_osm_features(
    aoi_poly,
    water_line_cfg["tags"],
    geom_types=("LineString", "MultiLineString"),
    keep_cols=water_line_cfg["keep_cols"],
)
water_polys = extract_osm_features(
    aoi_poly,
    water_poly_cfg["tags"],
    geom_types=("Polygon", "MultiPolygon"),
    keep_cols=water_poly_cfg["keep_cols"],
)

lac_cours_deau = gpd.GeoDataFrame(
    pd.concat([water_lines, water_polys], ignore_index=True),
    geometry="geometry",
    crs=EXPORT_CRS,
)
export_layer_gpkg(
    keep_columns(lac_cours_deau, water_poly_cfg["keep_cols"]),
    "lac_cours_deau",
    "osm_api",
)

for layer, layer_cfg in OSM_POLY_FEATURES.items():
    if layer == "lac_cours_deau":
        continue
    gdf = extract_osm_features(
        aoi_poly,
        layer_cfg["tags"],
        geom_types=("Polygon", "MultiPolygon"),
        keep_cols=layer_cfg["keep_cols"],
    )
    export_layer_gpkg(keep_columns(gdf, layer_cfg["keep_cols"]), layer, "osm_api")

## 11. Export des conflits préparés, si disponibles

Cette couche n'est pas recalculée ici. Elle est simplement reprise depuis `0_1` si elle existe.

In [ ]:
if conflict_zones is not None and not conflict_zones.empty:
    export_layer_gpkg(conflict_zones, "conflit_md", "prepared_conflict_zones")
else:
    LAYER_LOG.append({"layer": "conflit_md", "source": "prepared_conflict_zones", "n": 0, "exported": False})
    print("conflit_md: couche préparée absente ou vide")

## 12. Contrôle rapide

In [ ]:
layer_summary = pd.DataFrame(LAYER_LOG)
layer_summary